**Before running:** Runtime → Change runtime type → **T4 GPU**.

Always open this notebook fresh from GitHub (File → Open notebook → GitHub → `246mayank/cuda-sgemm`), not from Recent, so you get the latest cells. Run cells top to bottom with Shift+Enter.

In [ ]:
# 1 — environment
!nvidia-smi
!nvcc --version

In [ ]:
# 2 — fresh code
%cd /content
!rm -rf cuda-sgemm && git clone https://github.com/246mayank/cuda-sgemm
%cd /content/cuda-sgemm

In [ ]:
# 3 — build (prints registers/spills per kernel) and benchmark every kernel vs cuBLAS
!make
!./bench

In [ ]:
# 3b — measured DRAM bandwidth (memory roof for the roofline)
!./bandwidth

In [ ]:
# 4 — Nsight Compute. ncu locks clocks and flushes caches between passes, so only its counters are
#     used, never its timings. Each group is a separate command: if one metric name is unsupported,
#     only that group fails. Exit codes are printed so a failure is visible.
import datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d_%H%M%S')
# A: DRAM traffic (for measured arithmetic intensity), all kernels at N=2048
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_dram_{stamp}.csv --metrics dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "A exit: $?"
# B: cache behaviour, coalescing (sectors per request), achieved occupancy
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_mem_{stamp}.csv --metrics l1tex__t_sector_hit_rate.pct,lts__t_sector_hit_rate.pct,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum,sm__warps_active.avg.pct_of_peak_sustained_active ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "B exit: $?"
# C: shared-memory bank conflicts and instruction mix
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_smem_{stamp}.csv --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum,smsp__inst_executed_op_shared_ld.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__inst_executed.sum ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "C exit: $?"
# D: why is v0 3x faster at N=1023 than at N=1024? Same counters at both sizes
!ncu --csv --print-units base -k regex:sgemm_v0 --log-file results/ncu_v0stride_1023_{stamp}.csv --metrics dram__bytes_read.sum,l1tex__t_sector_hit_rate.pct,lts__t_sector_hit_rate.pct,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,gpu__time_duration.sum ./bench --kernel v0 --size 1023 --warmup 0 --reps 1 --no-write > /dev/null; echo "D1 exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v0 --log-file results/ncu_v0stride_1024_{stamp}.csv --metrics dram__bytes_read.sum,l1tex__t_sector_hit_rate.pct,lts__t_sector_hit_rate.pct,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,gpu__time_duration.sum ./bench --kernel v0 --size 1024 --warmup 0 --reps 1 --no-write > /dev/null; echo "D2 exit: $?"
!ls -la results/

In [ ]:
# 5 — bring results home
!zip -r results.zip results
from google.colab import files; files.download("results.zip")

Kaggle fallback: same cells; download results.zip from the Output panel instead of cell 5. Use `make ARCH=sm_60` if the GPU is a P100.